In [13]:
! pip install fiftyone

  Using cached deprecated-1.3.1-py2.py3-none-any.whl.metadata (5.9 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
   ---------- ----------------------------- 4.7/17.7 MB 31.8 MB/s eta 0:00:01
   ---------------------------- ----------- 12.8/17.7 MB 35.0 MB/s eta 0:00:01
   ---------------------------------------- 17.7/17.7 MB 33.9 MB/s  0:00:00
   ---------------------------------------- 0.0/25.7 MB ? eta -:--:--
   ---- ----------------------------------- 3.1/25.7 MB 46.5 MB/s eta 0:00:01
   ---- ----------------------------------- 3.1/25.7 MB 46.5 MB/s eta 0:00:01
   ---- ----------------------------------- 3.1/25.7 MB 46.5 MB/s eta 


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision import models
from torch.utils.data import DataLoader
# 추가된 부분
import fiftyone as fo
import fiftyone.zoo as foz

# 1. 환경 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device: ", device)

# 2. 데이터 전처리 (ResNet-50 표준 입력 크기 224x224)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# COCO Target을 분류용 Label(int)로 변환하는 함수
def coco_target_transform(target):
    # 첫 번째 객체의 category_id를 반환 (객체가 없으면 0번 배경 처리)
    if len(target) > 0:
        return target[0]['category_id']
    return 0

# 3. COCO 데이터셋 로드 (경로는 본인의 환경에 맞게 수정 필요)
# root: 이미지 폴더 경로, annFile: annotation json 파일 경로
dataset = foz.load_zoo_dataset(
    "coco-2017",
    split="validation",
    label_types=["detections"],
    classes=["person", "car"],
    max_samples=50,
)

# FiftyOne 데이터셋을 PyTorch Dataset 형식으로 변환
# (이때 transform을 적용합니다)
from fiftyone.utils.pytorch import FiftyOneTorchDataset

train_dataset = FiftyOneTorchDataset(dataset, transform=transform)
trainloader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0) 


# 4. ResNet-50 모델 생성
# weights=models.ResNet50_Weights.DEFAULT를 통해 최신 사전학습 가중치 로드
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

# 마지막 레이어 수정 (COCO category_id는 최대 90 내외지만 보통 80개 클래스 사용)
# 데이터셋의 실제 클래스 개수에 맞춰 수정하세요.
num_ftrs = model.fc.in_features
num_classes = 91 # COCO 기본 카테고리 수 기준
model.fc = nn.Linear(num_ftrs, num_classes)

model = model.to(device)

# 5. 손실 함수 및 최적화 설정
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

# 6. 모델 훈련 함수 (기존 코드와 동일 구조)
def train_model(model, trainloader, criterion, optimizer, epochs=5):
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for i, (inputs, labels) in enumerate(trainloader):
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            if i % 100 == 99:
                print(f'[{epoch + 1}, {i + 1}] loss: {running_loss / 100:.3f}')
                running_loss = 0.0
    print('Finished Training')

# 학습 실행
# train_model(model, trainloader, criterion, optimizer, epochs=2)

device:  cuda
 100% |██████|    1.9Gb/1.9Gb [6.4s elapsed, 0s remaining, 317.9Mb/s]       
Extracting annotations to 'C:\Users\csm02\fiftyone\coco-2017\raw\instances_val2017.json'
 100% |████████████████████| 50/50 [1.1s elapsed, 0s remaining, 47.3 images/s]         
Writing annotations for 50 downloaded samples to 'C:\Users\csm02\fiftyone\coco-2017\validation\labels.json'
Dataset info written to 'C:\Users\csm02\fiftyone\coco-2017\info.json'
Loading 'coco-2017' split 'validation'
 100% |███████████████████| 50/50 [478.1ms elapsed, 0s remaining, 105.1 samples/s]     
Dataset 'coco-2017-validation-50' created


ModuleNotFoundError: No module named 'fiftyone.utils.pytorch'

In [ ]:
from PIL import Image
import os
import matplotlib.pyplot as plt
import numpy as np

# 예측용 Transform (ImageNet 표준 가중치 기준)
inference_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

model.eval()
input_images = []
predicted_labels = []

for i in range(1, 11):
    file_name = f"./data/{i}.png" # 경로 확인 필요
    if not os.path.exists(file_name):
        continue

    img = Image.open(file_name).convert('RGB')
    img_tensor = inference_transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(img_tensor)
        _, predicted = torch.max(outputs, 1)
        # COCO 클래스 이름을 미리 정의해두었다면 리스트에서 가져오기
        # pred_class = coco_classes[predicted.item()] 
        predicted_labels.append(predicted.item())
        input_images.append(img_tensor.squeeze(0).cpu())

# 결과 시각화 (정규화 역산 포함)
plt.figure(figsize=(20, 8))
for i in range(len(input_images)):
    plt.subplot(2, 5, i + 1)
    img = input_images[i].numpy().transpose((1, 2, 0))
    # 정규화 역산: img * std + mean
    img = img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
    img = np.clip(img, 0, 1)
    plt.imshow(img)
    plt.title(f"Pred ID: {predicted_labels[i]}")
    plt.axis('off')
plt.show()

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\csm02/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:01<00:00, 44.1MB/s]
